In [49]:
import pandas as pd
import time
import numpy as np
import sys
sys.path.insert(0, "../../utils/")
from joblib import dump
import json
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split, cross_validate

In [50]:
MODELS = {
    "LinearRegression": LinearRegression,
    "SVM": SVR,
    "KNN": KNeighborsRegressor,
    "RandomForest": RandomForestRegressor,
    "AdaBoost": AdaBoostRegressor,
    "GradientBoosting": GradientBoostingRegressor,
    "XGBoost": xgb.XGBRegressor,
    "LGBM": lgb.LGBMRegressor,
}

In [51]:
GRIDS = {
    "RandomForest": {
        "n_estimators": [100, 500, 1000, 3000, 5000],
        "criterion": ["squared_error", "absolute_error"],
        "min_samples_split": [2, 10, 20],
        "min_samples_leaf": [1, 4, 8],
        "max_features": ["sqrt", "log2"],
        "max_depth": [10, 30, 50, None]
    },
    "AdaBoost": {
        "n_estimators": [50, 200, 500, 1000],
        "learning_rate": [0.01, 0.1, 1.0],
        "loss": ["linear", "square", "exponential"]
    },
    "GradientBoosting": {
        "n_estimators": [100, 300, 500],
        "learning_rate": [0.05, 0.1],
        "max_depth": [3, 5],
        "subsample": [0.8, 1.0],
        "loss": ["squared_error", "absolute_error"]
    },
    "XGBoost": {
        "n_estimators": [100, 300, 500],
        "max_depth": [3, 5],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "objective": ["reg:squarederror"]
    },
    "LGBM": {
        "n_estimators": [100, 300, 500],
        "max_depth": [3, 5, -1],
        "learning_rate": [0.01, 0.1],
        "num_leaves": [15, 31],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0],
        "objective": ["regression"]
    },
    "KNN": {
        "n_neighbors": [3, 5, 7],
        "weights": ["uniform", "distance"],
        "metric": ["euclidean", "manhattan"]
    },
    "SVM": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf"],
        "gamma": ["scale"]
    },
    "LinearRegression": {
        "fit_intercept": [True, False]
        }
}

In [52]:
def metrics(model, predict_val, y_val, dataset, div):
    mae_value = mean_absolute_error(y_pred=predict_val, y_true=y_val)
    mse_value = mean_squared_error(y_pred=predict_val, y_true=y_val)
    rmse_value = np.sqrt(mse_value)
    r2_value = r2_score(y_pred=predict_val, y_true=y_val)

    df_metrics = pd.DataFrame([[dataset, model, div, mae_value, mse_value, rmse_value, r2_value]],
        columns=["dataset", "model", "sampling", "MAE", "MSE", "RMSE", "R2"]
    )

    return df_metrics

In [53]:
def function_split(df_data, seed):
    #Separa los datos
    train_data, val_data = train_test_split(df_data, test_size=0.2, random_state=seed)
    return train_data, val_data

In [54]:
def cross_function(model, X_train, y_train, cv):
    scoring_metrics = {
        "neg_mean_squared_error": "neg_mean_squared_error",
        "neg_mean_absolute_error": "neg_mean_absolute_error",
        "r2": "r2",
        "explained_variance": "explained_variance"
    }

    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring_metrics, return_train_score=True)

    df_val = pd.DataFrame({metric: scores[f'test_{metric}'] for metric in scoring_metrics})
    df_val['fit_time'] = scores['fit_time']
    df_val['score_time'] = scores['score_time']
    df_val['Dataset'] = 'Validation'

    df_train = pd.DataFrame({metric: scores[f'train_{metric}'] for metric in scoring_metrics})
    df_train['fit_time'] = scores['fit_time']
    df_train['score_time'] = scores['score_time']
    df_train['Dataset'] = 'Train'

    results_metrics = pd.concat([df_val, df_train], ignore_index=True)

    return results_metrics

In [55]:
def get_model(model_name, seed):
    model_reg = MODELS[model_name]
    if model_name == "KNN":
        model = model_reg(n_jobs=-1)
    elif model_name in ["RandomForest", "XGBoost", "LGBM", "GradientBoosting"]:
        model = model_reg(random_state=seed, n_jobs=-1)
    elif model_name in ["SVM","LinearRegression"]:
        model = model_reg()
    else:
        model = model_reg(random_state=seed)
    return model

In [56]:
def function_train(model, model_name, train, val, div, seed):
    # Combina los conjuntos de entrenamiento y validación para la validación cruzada
    train_all= pd.concat([train, val], ignore_index=True)
    target = train_all["target"]
    train_all.drop(columns="target", inplace=True)

    print(f"Train {model_name} with seed {seed} and division {div}")    
    results = []

    #Valdación cruzada antes de la búsqueda de hiperparámetros
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
    cv_scores= cross_function(model, train_all, target, cv).copy()
    cv_scores["model"] = model_name
    cv_scores["sampling"] = div
    results.append(cv_scores)
    
    all_results = pd.concat(results, ignore_index=True)
    return all_results

In [57]:
def grid_function(model, model_name, train, val, seed, div):
    grid = GridSearchCV(estimator=model, param_grid=GRIDS[model_name], cv=10, scoring="f1_weighted", n_jobs=-1)
    X_train = train.drop(columns="target").values
    y_train = train["target"].values
    X_val= val.drop(columns="target").values
    y_val = val["target"].values

    print(f"GridSearchCV {model_name} with seed {seed} and division {div}")
    # Se realiza la búsqueda de hiperparámetros
    start_fit = time.time()
    grid.fit(X_train, y_train)
    elapsed_fit = time.time() - start_fit

    # Se obtienen los mejores parámetros y el mejor modelo
    best_model = grid.best_estimator_
    best_params = grid.best_params_
    best_score = grid.best_score_

    best_params = json.dumps(best_params, indent=4)
    with open(f"../../models/data/json/{model_name}_reg_Grid_{seed}_{div}_best_params.json", "w") as f:
        f.write(best_params)

    print(f"Best estimators: {best_model} and best parameters found: {best_params}")

    dump(best_model, f"../../models/data/best/{model_name}_reg_Grid_{seed}_{div}_best.joblib")

    # Se predice en el conjunto de entrenamiento y validación
    y_pred_train = best_model.predict(X_train)
    y_pred_val = best_model.predict(X_val)

    # Se guardan las métricas de entrenamiento y validación
    start_metrics = time.time()
    train_metrics = metrics(model_name, y_pred_train, y_train, "Train", div, predict_proba=y_proba_train)
    val_metrics = metrics(model_name, y_pred_val, y_val, "Validation", div, predict_proba=y_proba_val)
    elapsed_metrics = time.time() - start_metrics
    results = pd.concat([train_metrics, val_metrics], ignore_index=True)
    results['fit_time'] = elapsed_fit
    results['metrics_time'] = elapsed_metrics
    return results

In [58]:
def main_train(df_data, seed, model_name, grid_search=False):
    df_train, df_val = function_split(df_data, seed)
    model=get_model(model_name, seed)
    metrics=function_train(model, model_name, df_train, df_val, "Original", seed)
    metrics.to_csv(f"../../metrics/regression/{model_name}_reg_{seed}_metrics.csv", index=False)
    if grid_search:
        metrics_grid =grid_function(model, model_name, df_train, df_val, seed, "Original")
        metrics_grid.to_csv(f"../../metrics/regression/{model_name}_reg_{seed}_Grid_metrics.csv", index=False)    

In [59]:
repr_name="antiviral_homology_90_Q_prot5_embedding"
df_data = pd.read_csv(f"../../data/numerical_rep_reg/{repr_name}.csv")
df_data.drop(["experimental_characteristics"], axis=1, inplace=True)
seed= 42

In [60]:
for model_name in MODELS.keys():
    print(f"Processing {model_name} with {repr_name}")
    main_train(df_data, seed, model_name, grid_search=True)
    print(f"Finished {model_name}")
    print("=====================================")

Processing LinearRegression with antiviral_homology_90_Q_prot5_embedding
Train LinearRegression with seed 42 and division Original
GridSearchCV LinearRegression with seed 42 and division Original


c:\Users\hantr\anaconda3\envs\ML_Class\lib\site-packages\sklearn\model_selection\_search.py:1102: UserWarning: One or more of the test scores are non-finite: [nan nan]
  warnings.warn(


Best estimators: LinearRegression() and best parameters found: {
    "fit_intercept": true
}


NameError: name 'y_proba_train' is not defined